# Hindi voice-clone detection: robustness training

**How to run:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all. Allow Google Drive access when asked (tick every box). About 45 minutes the first time; later runs reuse what is saved on Drive.

Trains three versions of the detector with the same recipe and compares them:

- **A:** clean training audio only.
- **B:** every training clip plus one degraded copy: background noise from ESC-50 (half the time, at 5 to 20 dB) and a random voice-note or phone codec.
- **C:** B, plus SeamlessM4T fakes made from training sentences with voices not used in the test.

MMS-TTS stays out of training, so it remains a test on a system the detector never saw. Two new test conditions use ESC-50 noise recordings that are kept out of training: background noise at 10 dB, and the same followed by a GSM call.

Needs notebooks 01, 03 and 04 to have been run once (dataset backup, saved features, and the outside fakes on Drive).

## 1. Session setup

In [ ]:
import os, json, shutil, subprocess, importlib.util
import numpy as np, pandas as pd, soundfile as sf, torch
from sklearn.metrics import roc_curve
from tqdm.auto import tqdm

DRIVE = "/content/drive/MyDrive/voice-clone-detector"
OUT16 = "/content/data/hindi16k"
AASIST_DIR = "/content/aasist"
RESULTS = f"{DRIVE}/results"

# --- 1. Google Drive ---------------------------------------------------------
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULTS, exist_ok=True)

# --- 2. Dataset: restore from the Drive backup if this session does not have it
if not os.path.exists(f"{OUT16}/meta.csv"):
    backup = f"{DRIVE}/hindi16k.zip"
    assert os.path.exists(backup), "hindi16k.zip is not in your Drive folder. Stop here and tell Claude."
    print("restoring the dataset from Google Drive ...")
    shutil.copy(backup, "/content/hindi16k.zip")
    shutil.unpack_archive("/content/hindi16k.zip", OUT16)
meta = pd.read_csv(f"{OUT16}/meta.csv")
missing = [p for p in meta.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} clips are missing, for example {missing[0]}"

# --- 3. AASIST, the baseline detector ---------------------------------------
if not os.path.exists(AASIST_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/clovaai/aasist", AASIST_DIR], check=True)
spec = importlib.util.spec_from_file_location("aasist_model", f"{AASIST_DIR}/models/AASIST.py")
aasist = importlib.util.module_from_spec(spec)
spec.loader.exec_module(aasist)

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = json.load(open(f"{AASIST_DIR}/config/AASIST.conf"))["model_config"]
model = aasist.Model(cfg).to(device)
model.load_state_dict(torch.load(f"{AASIST_DIR}/models/weights/AASIST.pth", map_location=device))
model.eval()

# --- 4. Shared loading and scoring code -------------------------------------
CLIP_LEN = 64600      # about 4 seconds, the length AASIST was trained on
TARGET_DB = -23.0     # every clip is set to this loudness


def load_clip(path):
    a, sr = sf.read(path, dtype="float32")
    assert sr == 16000, f"{path} is {sr} Hz"
    a = a - a.mean()
    a = a * (10 ** (TARGET_DB / 20) / (np.sqrt(np.mean(a ** 2)) + 1e-9))
    if len(a) < CLIP_LEN:
        a = np.tile(a, CLIP_LEN // len(a) + 1)
    return a[:CLIP_LEN]


def score_clips(paths, batch=32, desc="scoring"):
    """Returns one number per clip. Higher means the detector thinks it is real."""
    realness, says_real = [], []
    for i in tqdm(range(0, len(paths), batch), desc=desc):
        x = torch.from_numpy(np.stack([load_clip(p) for p in paths[i:i + batch]])).to(device)
        with torch.no_grad():
            _, out = model(x)
        realness += out[:, 1].cpu().tolist()
        says_real += (out[:, 1] > out[:, 0]).cpu().tolist()
    return np.array(realness), np.array(says_real)


def eer(is_real, score):
    """Equal Error Rate in percent. 0 is perfect, 50 is a coin flip."""
    fpr, tpr, _ = roc_curve(is_real, score)
    i = np.nanargmin(np.abs((1 - tpr) - fpr))
    return 100 * (fpr[i] + 1 - tpr[i]) / 2


def eer_range(is_real, score, n=500, seed=0):
    """Rough 95% range for the EER, from re-drawing the test clips many times."""
    rng, vals = np.random.default_rng(seed), []
    for _ in range(n):
        j = rng.integers(0, len(score), len(score))
        if 0 < is_real[j].sum() < len(j):
            vals.append(eer(is_real[j], score[j]))
    return np.percentile(vals, [2.5, 97.5])


print("\n=== SETUP ===")
print("Running on:", "GPU" if device == "cuda" else "CPU (slow: switch the runtime to T4 GPU and run this again)")
print("Clips:", len(meta), "| test clips:", (meta.split == "test").sum())
print("Results folder:", RESULTS)
print("Files already there:", sorted(os.listdir(RESULTS)))

## 2. Compression conditions

In [ ]:
import os, subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

COND_DIR = "/content/data/hindi16k_conditions"
PHONE_BAND = "highpass=f=300,lowpass=f=3400"      # the narrow band a phone line carries

# name: (how ffmpeg encodes it, file type)
CONDITIONS = {
    "opus_24k":   (["-c:a", "libopus", "-b:a", "24k", "-application", "voip"], "ogg"),   # voice note
    "opus_12k":   (["-c:a", "libopus", "-b:a", "12k", "-application", "voip"], "ogg"),   # voice note, poor connection
    "phone_g711": (["-af", PHONE_BAND, "-ar", "8000", "-c:a", "pcm_alaw"], "wav"),       # landline / internet call
    "phone_gsm":  (["-af", PHONE_BAND, "-ar", "8000", "-c:a", "libgsm"], "gsm"),         # 2G mobile call
}

have = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
for name in list(CONDITIONS):
    codec = CONDITIONS[name][0][CONDITIONS[name][0].index("-c:a") + 1]
    if f" {codec} " not in have:
        print(f"skipping {name}: this ffmpeg has no {codec} encoder")
        del CONDITIONS[name]


def degrade(src, dst, args, ext):
    """Encode a clip the way a call or voice note would, then decode it back to 16 kHz WAV."""
    if os.path.exists(dst):
        return
    with tempfile.TemporaryDirectory() as tmp:
        mid = f"{tmp}/x.{ext}"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", src, "-ac", "1", *args, mid], check=True)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", mid, "-ac", "1", "-ar", "16000",
                        "-c:a", "pcm_s16le", dst + ".tmp.wav"], check=True)
    os.replace(dst + ".tmp.wav", dst)


test = meta[meta.split == "test"].reset_index(drop=True)
test["is_real"] = (test.label == "real").astype(int)

# --- 1. Make a degraded copy of every test clip, for every condition ---------
paths = {"clean": list(test.path)}
for name, (args, ext) in CONDITIONS.items():
    for label in ("fake", "real"):
        os.makedirs(f"{COND_DIR}/{name}/{label}", exist_ok=True)
    paths[name] = [f"{COND_DIR}/{name}/{label}/{os.path.basename(p)}" for p, label in zip(test.path, test.label)]
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
        jobs = [pool.submit(degrade, s, d, args, ext) for s, d in zip(test.path, paths[name])]
        for j in tqdm(jobs, desc=f"encoding {name}"):
            j.result()

# --- 2. Score every condition with AASIST ------------------------------------
rows, scores = [], []
for name, plist in paths.items():
    realness, says_real = score_clips(plist, desc=f"scoring {name}")
    d = test[["path", "label", "tool", "speaker_id", "is_real"]].assign(
        condition=name, cond_path=plist, realness=realness, says_real=says_real)
    scores.append(d)
    real = d[d.label == "real"]
    row = {"condition": name, "EER all": eer(d.is_real.to_numpy(), d.realness.to_numpy())}
    for t in ("freevc24", "xtts_v2"):
        sub = pd.concat([real, d[d.tool == t]])
        row[f"EER {t}"] = eer(sub.is_real.to_numpy(), sub.realness.to_numpy())
    row["real called fake %"] = 100 * (1 - real.says_real.mean())
    for t in ("freevc24", "xtts_v2"):
        row[f"{t} passed as real %"] = 100 * d[d.tool == t].says_real.mean()
    rows.append(row)

summary = pd.DataFrame(rows).set_index("condition").round(1)
pd.concat(scores).to_csv(f"{RESULTS}/aasist_conditions_hindi_scores.csv", index=False)
summary.to_csv(f"{RESULTS}/aasist_conditions_hindi_summary.csv")

print("\n=== AASIST on Hindi test clips, clean and degraded ===")
print(summary.to_string())
print(f"\n{len(test)} test clips per condition. Saved to {RESULTS}")

## 3. The clean-trained detector

Reuses the features saved on Drive, so this is quick.

In [ ]:
import copy
import torch.nn as nn
from transformers import Wav2Vec2Model

assert "paths" in globals() and "clean" in paths, "Run the Step 8 cell first: it makes the degraded test clips."

SSL_NAME = "facebook/wav2vec2-xls-r-300m"     # speech model pretrained on 128 languages
FEAT_DIR = f"{DRIVE}/features"      # saved to Drive so a reset does not cost the slow part
MODELS = f"{DRIVE}/models"
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

# --- 1. Turn every clip into features with the pretrained speech model -------
ssl_model = Wav2Vec2Model.from_pretrained(SSL_NAME).to(device).eval()


def embed(path_list, name):
    """For each clip: one vector per layer of the speech model, averaged over time."""
    cache = f"{FEAT_DIR}/{name}.npy"
    if os.path.exists(cache):
        return np.load(cache)
    out = []
    for i in tqdm(range(0, len(path_list), 32), desc=f"features {name}"):
        x = np.stack([load_clip(p) for p in path_list[i:i + 32]])
        x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-7)
        with torch.no_grad():
            layers = ssl_model(torch.from_numpy(x).to(device), output_hidden_states=True).hidden_states
        out.append(torch.stack([h.mean(1) for h in layers], 1).cpu().numpy().astype("float16"))
    out = np.concatenate(out)
    assert np.isfinite(out).all(), "features contain NaN or infinity"
    np.save(cache, out)
    return out


F_all = embed(list(meta.path), "clean_all")
is_real_all = (meta.label == "real").to_numpy().astype(int)
tool_all, split_all = meta.tool.to_numpy(), meta.split.to_numpy()
test_mask = split_all == "test"
assert list(meta.path[test_mask]) == paths["clean"], "test clips are not in the expected order"
F_test = {"clean": F_all[test_mask]}
for name in paths:
    if name != "clean":
        F_test[name] = embed(paths[name], f"test_{name}")
y_test, tool_test = is_real_all[test_mask], tool_all[test_mask]


# --- 2. A small classifier on top of those features --------------------------
class Head(nn.Module):
    """Learns how much to trust each layer, then draws one line between real and fake."""

    def __init__(self, n_layers, dim):
        super().__init__()
        self.layer_weights = nn.Parameter(torch.zeros(n_layers))
        self.drop = nn.Dropout(0.2)
        self.out = nn.Linear(dim, 1)

    def forward(self, f):
        w = torch.softmax(self.layer_weights, 0)[None, :, None]
        return self.out(self.drop((w * f).sum(1))).squeeze(-1)


def eer_threshold(is_real, score):
    fpr, tpr, thr = roc_curve(is_real, score)
    return thr[np.nanargmin(np.abs((1 - tpr) - fpr))]


def train_head(tools, epochs=60, seed=0):
    """Train on real clips plus fakes from the given tools. Returns a scoring function and details."""
    torch.manual_seed(seed)
    use = np.isin(tool_all, ["real", *tools])
    tr, va = use & (split_all == "train"), use & (split_all == "val")
    mu = torch.tensor(F_all[tr].astype("float32").mean(0, keepdims=True), device=device)
    sd = torch.tensor(F_all[tr].astype("float32").std(0, keepdims=True) + 1e-5, device=device)
    prep = lambda F: (torch.tensor(F.astype("float32"), device=device) - mu) / sd
    Xtr, ytr = prep(F_all[tr]), torch.tensor(is_real_all[tr], dtype=torch.float32, device=device)
    Xva, yva = prep(F_all[va]), is_real_all[va]

    head = Head(F_all.shape[1], F_all.shape[2]).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=(1 - ytr).sum() / ytr.sum())
    best = (101.0, None, 0)
    for epoch in range(1, epochs + 1):
        head.train()
        order = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(order), 64):
            j = order[i:i + 64]
            opt.zero_grad()
            loss_fn(head(Xtr[j]), ytr[j]).backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            val_eer = eer(yva, head(Xva).cpu().numpy())
        if val_eer < best[0]:
            best = (val_eer, copy.deepcopy(head.state_dict()), epoch)
    head.load_state_dict(best[1])
    head.eval()

    def score(F):
        with torch.no_grad():
            return head(prep(F)).cpu().numpy()

    return dict(score=score, head=head, mu=mu, sd=sd, val_eer=best[0], epoch=best[2],
                threshold=eer_threshold(yva, score(F_all[va])), n_train=int(tr.sum()))


def eer_vs_real(scores, tool):
    keep = (tool_test == "real") | (tool_test == tool)
    return eer(y_test[keep], scores[keep])


# --- 3. Train on clean Hindi (both tools), test on clean and degraded audio --
both = train_head(["freevc24", "xtts_v2"])
rows, out_scores = [], []
for name, F in F_test.items():
    s = both["score"](F)
    says_real = s >= both["threshold"]
    row = {"condition": name, "EER all": eer(y_test, s), "EER freevc24": eer_vs_real(s, "freevc24"),
           "EER xtts_v2": eer_vs_real(s, "xtts_v2"),
           "real called fake %": 100 * (1 - says_real[y_test == 1].mean()),
           "fake passed as real %": 100 * says_real[y_test == 0].mean()}
    if "summary" in globals() and name in summary.index:
        row["AASIST EER all"] = summary.loc[name, "EER all"]
    rows.append(row)
    out_scores.append(pd.DataFrame(dict(path=paths[name], condition=name, tool=tool_test, is_real=y_test,
                                        realness=s, says_real=says_real)))
table_a = pd.DataFrame(rows).set_index("condition").round(1)

# --- 4. Same model, but trained on one cloning tool and tested on the other --
rows = [{"trained on": "both tools", "EER freevc24": eer_vs_real(both["score"](F_test["clean"]), "freevc24"),
         "EER xtts_v2": eer_vs_real(both["score"](F_test["clean"]), "xtts_v2"), "unseen tool": "none"}]
for seen, unseen in (("freevc24", "xtts_v2"), ("xtts_v2", "freevc24")):
    one = train_head([seen])
    s = one["score"](F_test["clean"])
    rows.append({"trained on": f"{seen} only", "EER freevc24": eer_vs_real(s, "freevc24"),
                 "EER xtts_v2": eer_vs_real(s, "xtts_v2"), "unseen tool": unseen})
table_b = pd.DataFrame(rows).set_index("trained on").round(1)

# --- 5. Save and report -------------------------------------------------------
pd.concat(out_scores).to_csv(f"{RESULTS}/xlsr_clean_trained_scores.csv", index=False)
table_a.to_csv(f"{RESULTS}/xlsr_clean_trained_by_condition.csv")
table_b.to_csv(f"{RESULTS}/xlsr_clean_trained_unseen_tool.csv")
torch.save({"head": both["head"].state_dict(), "mu": both["mu"].cpu(), "sd": both["sd"].cpu(),
            "threshold": float(both["threshold"]), "ssl_name": SSL_NAME}, f"{MODELS}/xlsr_head_clean.pt")

lw = torch.softmax(both["head"].layer_weights, 0).detach().cpu().numpy()
print("\n=== YOUR DETECTOR, trained on clean Hindi only ===")
print(f"Trained on {both['n_train']} clips. Validation EER {both['val_eer']:.1f}% (best at epoch {both['epoch']}).")
print("Layers it relies on most:", ", ".join(f"{i} ({100 * lw[i]:.0f}%)" for i in np.argsort(-lw)[:3]),
      f"of {len(lw)}")
print("\nA. Tested on clean and degraded audio (decisions use one fixed threshold set on clean validation clips):")
print(table_a.to_string())
print("\nB. Clean audio, when one cloning tool was never seen in training:")
print(table_b.to_string())
print(f"\nSaved to {RESULTS} and {MODELS}")

## 4. Robustness training and comparison

In [ ]:
import os, glob, subprocess, zlib
from math import gcd
from concurrent.futures import ThreadPoolExecutor
from scipy.signal import resample_poly
from transformers import AutoProcessor, SeamlessM4Tv2ForTextToSpeech

assert "train_head" in globals() and "paths" in globals(), "Run the earlier cells first: setup, compression and training."
WORK = "/content/data/robust"
ESC_DIR = "/content/data/ESC-50"
os.makedirs(WORK, exist_ok=True)


def embed_checked(path_list, name):
    """Like embed(), but recomputes if a saved file does not match the clip list."""
    cache = f"{FEAT_DIR}/{name}.npy"
    if os.path.exists(cache) and len(np.load(cache, mmap_mode="r")) != len(path_list):
        os.remove(cache)
    return embed(path_list, name)


# --- 1. Background noise: ESC-50 environmental recordings --------------------
if not os.path.isdir(f"{ESC_DIR}/audio"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/karolpiczak/ESC-50", ESC_DIR], check=True)


def load_noise(path):
    a, sr = sf.read(path, dtype="float32", always_2d=True)
    a = a.mean(axis=1)
    g = gcd(sr, 16000)
    return resample_poly(a, 16000 // g, sr // g).astype("float32")


def steady(path):
    """Keep recordings where sound is present most of the time (rain, traffic, crowds), not single bangs."""
    a = load_noise(path)
    return np.mean(np.abs(a) > 1e-4) > 0.8 and np.sqrt(np.mean(a ** 2)) > 1e-3


noise_ok = [p for p in tqdm(sorted(glob.glob(f"{ESC_DIR}/audio/*.wav")), desc="checking noise") if steady(p)]
noise_train = [p for p in noise_ok if os.path.basename(p)[0] in "1234"]     # ESC-50 folds 1 to 4
noise_test = [p for p in noise_ok if os.path.basename(p)[0] == "5"]         # fold 5, never used in training
print(f"{len(noise_train)} noise recordings for training, {len(noise_test)} kept apart for testing")
assert len(noise_train) >= 20 and len(noise_test) >= 5, "too few usable noise recordings. Paste this to Claude."


def mix_noise(src, dst, noise_path, snr_db, seed):
    s, _ = sf.read(src, dtype="float32")
    n = load_noise(noise_path)
    if len(n) < len(s):
        n = np.tile(n, len(s) // len(n) + 1)
    start = np.random.default_rng(seed).integers(0, len(n) - len(s) + 1)
    n = n[start:start + len(s)]
    x = s + n * np.sqrt((np.mean(s ** 2) + 1e-12) / ((np.mean(n ** 2) + 1e-12) * 10 ** (snr_db / 10)))
    peak = np.abs(x).max()
    if peak > 0.99:
        x = x * (0.99 / peak)
    sf.write(dst + ".tmp.wav", x.astype("float32"), 16000, subtype="PCM_16")
    os.replace(dst + ".tmp.wav", dst)


def make_version(src, dst, noise, codec, seed):
    """noise: (noise file, SNR in dB) or None. codec: a name from CONDITIONS or None."""
    if os.path.exists(dst):
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if noise and codec:
        mid = dst + ".noisy.wav"
        mix_noise(src, mid, *noise, seed)
        degrade(mid, dst, *CONDITIONS[codec])
        os.remove(mid)
    elif noise:
        mix_noise(src, dst, *noise, seed)
    else:
        degrade(src, dst, *CONDITIONS[codec])


def run_jobs(jobs, desc):
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
        for f in tqdm([pool.submit(make_version, *j) for j in jobs], desc=desc):
            f.result()
    return [j[1] for j in jobs]


CODECS = [None] + list(CONDITIONS)


def degraded_copies(srcs, tag):
    """One copy of each clip with random noise (half the time) and a random codec, never both left out."""
    jobs, base = [], zlib.crc32(tag.encode())
    for i, src in enumerate(srcs):
        r = np.random.default_rng(base + i)
        noise = (noise_train[r.integers(len(noise_train))], float(r.uniform(5, 20))) if r.random() < 0.5 else None
        codec = CODECS[r.integers(len(CODECS))]
        if noise is None and codec is None:
            noise = (noise_train[r.integers(len(noise_train))], float(r.uniform(5, 20)))
        jobs.append((src, f"{WORK}/{tag}/{i:05d}.wav", noise, codec, base + i))
    return run_jobs(jobs, f"making {tag}")


# --- 2. More variety of fakes: SeamlessM4T voices and sentences not in the test --
def sentences(split, n):
    return sorted({t for t in meta[(meta.split == split) & (meta.label == "real")].text.dropna()
                   if 20 <= len(t) <= 150})[:n]


SEAM = {"train": (f"{DRIVE}/outside_fakes/seamless_train", sentences("train", 400)),
        "val": (f"{DRIVE}/outside_fakes/seamless_val", sentences("val", 100))}
seam_clips = {}
try:
    if any(len(glob.glob(f"{d}/*.wav")) + len(glob.glob(f"{d}/*.skip")) < len(t) for d, t in SEAM.values()):
        proc = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
        smt = SeamlessM4Tv2ForTextToSpeech.from_pretrained(
            "facebook/seamless-m4t-v2-large",
            torch_dtype=torch.float16 if device == "cuda" else torch.float32).to(device).eval()
        for split, (folder, texts_) in SEAM.items():
            os.makedirs(folder, exist_ok=True)
            for i, text in enumerate(tqdm(texts_, desc=f"generating SeamlessM4T {split}")):
                path = f"{folder}/{os.path.basename(folder)}_{i:04d}.wav"
                if os.path.exists(path) or os.path.exists(path + ".skip"):
                    continue
                inputs = proc(text=text, src_lang="hin", return_tensors="pt").to(device)
                with torch.no_grad():                         # voices 10, 30, ..., 190: none used in the test
                    wav = smt.generate(**inputs, tgt_lang="hin", speaker_id=(i % 10) * 20 + 10)[0]
                a = wav.float().cpu().numpy().squeeze()
                sr = smt.config.sampling_rate
                if sr != 16000:
                    g = gcd(sr, 16000)
                    a = resample_poly(a, 16000 // g, sr // g)
                if len(a) >= 16000:
                    sf.write(path, np.clip(a, -1, 1).astype("float32"), 16000, subtype="PCM_16")
                else:
                    open(path + ".skip", "w").close()
        del smt
        if device == "cuda":
            torch.cuda.empty_cache()
    seam_clips = {split: sorted(glob.glob(f"{folder}/*.wav")) for split, (folder, _) in SEAM.items()}
except Exception as e:
    print("SeamlessM4T training fakes skipped, so version C is left out:", repr(e)[:300])

# --- 3. Training and validation features: every clip, plus one degraded copy --
tr, va = split_all == "train", split_all == "val"
use = np.isin(tool_all, ["real", "freevc24", "xtts_v2"])
tr, va = tr & use, va & use
F_tr_aug = embed_checked(degraded_copies(list(meta.path[tr]), "aug_train"), "robust_aug_train")
F_va_aug = embed_checked(degraded_copies(list(meta.path[va]), "aug_val"), "robust_aug_val")
y_tr, y_va = is_real_all[tr], is_real_all[va]
if seam_clips:
    F_seam = {s: embed_checked(seam_clips[s], f"seamless_{s}_clean") for s in seam_clips}
    F_seam_aug = {s: embed_checked(degraded_copies(seam_clips[s], f"aug_seamless_{s}"), f"robust_aug_seamless_{s}")
                  for s in seam_clips}


# --- 4. Two new test conditions: background noise, and noise plus a GSM call --
def noisy_versions(srcs, tag):
    out = {}
    for cond, codec in (("noisy", None), ("noisy_gsm", "phone_gsm")):
        jobs, base = [], zlib.crc32(tag.encode())
        for i, src in enumerate(srcs):
            r = np.random.default_rng(base + i)
            noise = (noise_test[r.integers(len(noise_test))], 10.0)        # unseen noise at 10 dB
            jobs.append((src, f"{WORK}/test_{cond}/{tag}/{i:05d}.wav", noise, codec, base + i))
        out[cond] = run_jobs(jobs, f"making {tag} {cond}")
    return out


test_paths = dict(paths)
test_paths.update(noisy_versions(paths["clean"], "test"))
F_eval = {name: (F_test[name] if name in F_test else embed_checked(p, f"test_{name}")) for name, p in test_paths.items()}
CONDS = list(test_paths)


def outside_features(folder, short):
    """Fakes from notebook 04, in every condition. Reuses the features notebook 04 saved."""
    clips = sorted(glob.glob(f"{folder}/*.wav"))
    if not clips:
        return None
    feats = {"clean": embed(clips, f"outside_{short}_clean")}
    for name in paths:
        if name != "clean":
            dsts = [f"{COND_DIR}/{name}/outside/{os.path.basename(p)}" for p in clips]
            if not os.path.exists(f"{FEAT_DIR}/outside_{short}_{name}.npy"):
                os.makedirs(os.path.dirname(dsts[0]), exist_ok=True)
                for s_, d_ in zip(clips, dsts):
                    degrade(s_, d_, *CONDITIONS[name])
            feats[name] = embed(dsts, f"outside_{short}_{name}")
    for cond, p in noisy_versions(clips, short).items():
        feats[cond] = embed_checked(p, f"outside_{short}_{cond}")
    return feats


outside_eval = {"MMS-TTS": outside_features(f"{DRIVE}/outside_fakes/mms", "mms-tts"),
                "SeamlessM4T": outside_features(f"{DRIVE}/outside_fakes/seamless", "seamlessm4t")}
other_real = {k: np.load(f"{FEAT_DIR}/{f}.npy") for k, f in (("FLEURS", "other_fleurs"), ("IndicTTS", "other_indictts"))
              if os.path.exists(f"{FEAT_DIR}/{f}.npy")}


# --- 5. Train three versions with the same recipe ------------------------------
def fit(Xtr_np, ytr_np, Xva_np, yva_np, epochs=40, seed=0):
    torch.manual_seed(seed)
    mu = torch.tensor(Xtr_np.astype("float32").mean(0, keepdims=True), device=device)
    sd = torch.tensor(Xtr_np.astype("float32").std(0, keepdims=True) + 1e-5, device=device)
    prep = lambda F: (torch.tensor(F.astype("float32"), device=device) - mu) / sd
    Xtr, ytr = prep(Xtr_np), torch.tensor(ytr_np, dtype=torch.float32, device=device)
    Xva, yva = prep(Xva_np), torch.tensor(yva_np, dtype=torch.float32, device=device)
    head = Head(Xtr_np.shape[1], Xtr_np.shape[2]).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=(1 - ytr).sum() / ytr.sum())
    best = (float("inf"), None, 0)
    for epoch in range(1, epochs + 1):
        head.train()
        order = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(order), 64):
            j = order[i:i + 64]
            opt.zero_grad()
            loss_fn(head(Xtr[j]), ytr[j]).backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            val_loss = loss_fn(head(Xva), yva).item()
        if val_loss < best[0]:
            best = (val_loss, copy.deepcopy(head.state_dict()), epoch)
    head.load_state_dict(best[1])
    head.eval()

    def score(F):
        with torch.no_grad():
            return head(prep(F)).cpu().numpy()

    va_scores = score(Xva_np)
    return dict(score=score, head=head, mu=mu, sd=sd, epoch=best[2], val_eer=eer(yva_np, va_scores),
                threshold=eer_threshold(yva_np, va_scores), n_train=len(ytr_np))


cat = np.concatenate
versions = {
    "A: clean audio": fit(F_all[tr], y_tr, F_all[va], y_va),
    "B: + noise and phone audio": fit(cat([F_all[tr], F_tr_aug]), cat([y_tr, y_tr]),
                                      cat([F_all[va], F_va_aug]), cat([y_va, y_va])),
}
if seam_clips:
    z = lambda s: np.zeros(len(F_seam[s]), dtype=int)
    versions["C: B + SeamlessM4T fakes"] = fit(
        cat([F_all[tr], F_tr_aug, F_seam["train"], F_seam_aug["train"]]),
        cat([y_tr, y_tr, z("train"), z("train")]),
        cat([F_all[va], F_va_aug, F_seam["val"], F_seam_aug["val"]]),
        cat([y_va, y_va, z("val"), z("val")]))


# --- 6. Compare them ------------------------------------------------------------
def real_vs(m, cond, F_fake):
    s_real = m["score"](F_eval[cond])[y_test == 1]
    s_fake = m["score"](F_fake)
    return eer(np.r_[np.ones(len(s_real)), np.zeros(len(s_fake))], np.r_[s_real, s_fake])


rows = []
nice = {"clean": "clean", "opus_12k": "voice note 12k", "phone_gsm": "GSM call", "noisy": "background noise",
        "noisy_gsm": "noise + GSM call"}
for cond in ("clean", "phone_gsm", "noisy", "noisy_gsm"):
    rows.append(("Catching fakes", f"IndicSynth fakes, EER, {nice[cond]}",
                 {k: real_vs(m, cond, F_eval[cond][y_test == 0]) for k, m in versions.items()}))
for sys_name, feats in outside_eval.items():
    if feats is None:
        continue
    note = "never in training" if sys_name == "MMS-TTS" else "other voices in C's training"
    for cond in ("clean", "phone_gsm", "noisy_gsm"):
        rows.append(("Catching fakes", f"{sys_name} ({note}), EER, {nice[cond]}",
                     {k: real_vs(m, cond, feats[cond]) for k, m in versions.items()}))
    for cond in ("clean", "phone_gsm", "noisy_gsm"):
        rows.append(("Catching fakes", f"{sys_name} passed as real %, {nice[cond]}",
                     {k: 100 * (m["score"](feats[cond]) >= m["threshold"]).mean() for k, m in versions.items()}))
for cond in ("clean", "opus_12k", "phone_gsm", "noisy", "noisy_gsm"):
    rows.append(("False alarms", f"Kathbath real called fake %, {nice[cond]}",
                 {k: 100 * (m["score"](F_eval[cond][y_test == 1]) < m["threshold"]).mean() for k, m in versions.items()}))
for name, F in other_real.items():
    rows.append(("False alarms", f"{name} real called fake %, clean",
                 {k: 100 * (m["score"](F) < m["threshold"]).mean() for k, m in versions.items()}))

compare = pd.DataFrame([{"check": c, "measure": n, **v} for c, n, v in rows]).set_index(["check", "measure"]).round(1)
compare.to_csv(f"{RESULTS}/robust_training_comparison.csv")
for k, m in versions.items():
    torch.save({"head": m["head"].state_dict(), "mu": m["mu"].cpu(), "sd": m["sd"].cpu(),
                "threshold": float(m["threshold"]), "ssl_name": SSL_NAME},
               f"{MODELS}/xlsr_head_{k.split(':')[0].lower()}.pt")

print("\n=== ROBUST TRAINING: three versions of the detector ===")
for k, m in versions.items():
    print(f"{k}: {m['n_train']} training clips, best at epoch {m['epoch']}, validation EER {m['val_eer']:.1f}%")
print("\nAll numbers in %. Lower is better everywhere.\n")
with pd.option_context("display.width", 250, "display.max_colwidth", 70):
    print(compare.to_string())
print(f"\nSaved to {RESULTS} and {MODELS}")